# Methodology Update Report
**Branch:** `methodology-improvements`  
**Date:** April 2026  
**Authors:** Giacomo Roversi, with Claude Code review

This notebook documents the rationale behind the methodological changes introduced in this branch, for internal reference. It is **not** the final public-facing report (see `report.ipynb` when ready).

---
## 1. Motivation: why the original pipeline needed updating

The original clustering (on `main`) used four features: `land_flag`, `ice_water_path` (IWP), `liquid_water_path` (LWP), `aerosol_optical_thickness_355nm` (AOT), normalised with min-max scaling.

Three issues were identified:

### 1.1 `land_flag` was distorting the clusters
`land_flag` is the fraction of 1-second observations within each 1-minute bin that were acquired over land. It is a **geographic/surface property**, not a physical cloud-aerosol variable. Including it gave the algorithm a shortcut to separate land from ocean, producing at least one cluster that was essentially a "land pixel" cluster rather than an aerosol-cloud regime. Since all three study regions are predominantly ocean (the bounding boxes were chosen for open-ocean conditions), land pixels are a small artefact of the rectangular box — not scientifically interesting.

**Fix:** filter `land_flag < 0.1` before clustering (≥90% of the window must be over ocean), then drop the variable entirely from the feature matrix.

### 1.2 Min-max scaling on heavy-tailed distributions
IWP, LWP, and AOT are approximately **log-normal**: most observations cluster near zero, but rare events (deep convective towers, dust plumes) reach values orders of magnitude higher. Applied directly, min-max scaling compresses the bulk of the data into a narrow band near 0, while a handful of extreme values anchor the maximum. The result is that K-means geometry is dominated by those outliers, and the cluster boundaries reflect outlier distances rather than typical atmospheric conditions.

**Fix:** apply `np.log1p` before min-max. `log1p(x) = log(1+x)` is safe at zero, compresses the upper tail, and spreads the bulk of the distribution across the full [0,1] range after normalisation.

### 1.3 Antimeridian crossing for Antarctica
The Ross Sea region straddles the 180°/−180° longitude boundary. The original code split it into two bounding boxes (R1: 160–180°E, R2: 180–140°W) and concatenated the resulting datasets. This is functionally correct but fragile: any bbox filtering applied later must replicate the split logic.

**Fix (in `kmeans_AN.ipynb`):** convert longitudes to the 0–360° convention before filtering, so the Ross Sea sits in a single contiguous box (160–220°E).

---
## 2. Changes applied — file by file

### 2.1 `merger.ipynb` — data preparation (West Pacific)

| Cell / location | Change | Reason |
|---|---|---|
| Imports | Removed duplicate `from tqdm import tqdm` | Redundant import |
| `_test = ''` | Added explanatory comment | Name with `_` prefix looked like dead code |
| `.clip(min=0, max=1e2)` | Added comment explaining the 100-unit cap | Magic number with no context |
| Cell `10554/12/5/30` | Deleted | Dead exploratory expression — no assignment, no output used |
| `mode_func` | Added full docstring | Function had only an inline comment; parameters and return type were undocumented |

### 2.2 `kmeans.ipynb` — clustering (West Pacific)

| Cell / location | Change | Reason |
|---|---|---|
| Imports | Removed duplicate `from tqdm import tqdm` | Redundant import |
| `cluster_variance` | Removed 3 unused variables (`outputs`, `variance`, `kmeans`); `verbose=2→0`; returns `(variances, K)` not `(variances, K, n)`; added docstring | Dead code + noisy output + redundant return value |
| Bbox filter | Added `& (ds.land_flag < 0.1)` | Exclude land-contaminated minutes (see §1.1) |
| Feature matrix | Dropped `land_flag`; added `np.log1p` before min-max; now 3 features | See §1.1 and §1.2 |
| `vars` dict | Renamed to `feature_cols`; updated indices to match 3-feature matrix | `vars` shadows Python builtin; indices shifted after removing `land_flag` |
| `plot_kmeans` | `y_kmeans` and `kmeans_model` now explicit parameters; `if not ax` → `if ax is None`; centroid loop uses `len(centers)` not hardcoded `range(5)`; added docstring | Global dependency hidden in function signature; `not ax` raises `ValueError` on Axes objects; hardcoded cluster count |
| All `plot_kmeans` call sites | Passed `y_kmeans` and `kmeans` explicitly | Required by new signature |
| Histogram loop | `range(4)` → `range(k)` | Bug: 5th cluster column was never plotted |
| `ds_tc` filter | Added same `land_flag < 0.1` filter | Labels must align with the filtered numerical data |
| Regression print | `print(q, m, n)` → `print(f"intercept={q:.4f}, slope={m:.4f}")` | `n` was a leftover from old 3-tuple return; formatted output is clearer |
| Index-based feature access | `x[indices, 3]` / `x[indices, 1]` → `x[indices, feature_cols['aot']]` etc. | Hardcoded indices break silently when column order changes |

### 2.3 `kmeans_EP.ipynb` — clustering (East Pacific)

Same changes as §2.2 applied symmetrically. Additionally:

| Cell / location | Change | Reason |
|---|---|---|
| AC__TC__2B section | Added `plot_colors`, `labels`, `cmap/bounds/norm` setup + histogram grid | Section existed only as a `# TODO` comment — completed to match WP notebook |

### 2.4 `kmeans_AN.ipynb` — clustering (Antarctica / Ross Sea) *(to create)*

Planned additions:

| Step | Details |
|---|---|
| Load data | `challenge_1min_numerical_AN.nc` (or `_R1` + `_R2` if separate) |
| Antimeridian fix | Convert lon to 0–360 (`lon % 360`) before bbox filter; single box 160–220°E, 60–80°S |
| Ocean filter | Same `land_flag < 0.1` — Antarctic coast pixels should be excluded |
| Feature matrix | Same `log1p` + min-max on IWP, LWP, AOT |
| Elbow + clustering | Re-run; Antarctica may prefer a different k (expect fewer cloud types) |
| AC__TC__2B cross-ref | Full histogram grid as in WP and EP |

---
## 4. `kmeans_AN.ipynb` — Antarctica (Ross Sea)

Notebook creato da zero. Differenze rispetto a WP ed EP:

| Step | Dettaglio |
|---|---|
| Load data | `challenge_1min_numerical_AN.nc` |
| Antimeridian fix | `lon360 = ds.longitude % 360`; bbox unico 160–220°E, 80–60°S (unifica R1 e R2) |
| Ocean filter | `land_flag < 0.1` — esclude pixel costieri |
| Feature matrix | identica a WP/EP: `log1p` + min-max su IWP, LWP, AOT |
| Elbow + clustering | k=5 come default; rieseguire elbow dopo il `log1p` per confermare |
| AC__TC__2B cross-ref | griglia istogrammi completa; carica `challenge_1min_complete_AN.nc` con lo stesso filtro antimeridiano |

**Nota:** se i file `_AN.nc` non sono ancora sul server, rieseguire `merger_AN.ipynb` (o il merger equivalente) prima di questo notebook.

---
## 3. Roadmap toward the final report

```
Done ✓
  merger.ipynb          — code quality + docstrings
  kmeans.ipynb (WP)     — methodology fix + code quality
  kmeans_EP.ipynb       — methodology fix + AC__TC section completed

Next →
  kmeans_AN.ipynb       — create from scratch with antimeridian fix

Then →
  report.ipynb          — narrative + figures
    ├─ Three-region comparison  (WP / EP / AN side-by-side)
    ├─ Cluster composition vs AC__TC__2B  (quantitative, per region)
    └─ Interpretation + conclusions
```

### Open questions before `report.ipynb`
- Are `challenge_1min_*_AN.nc` files available on the server, or does `merger_AN.ipynb` need to be re-run?
- Is k=5 still the elbow choice after the `log1p` transformation? Re-run elbow plots on all three regions before fixing k.
- For the AC__TC__2B cross-reference: present as **percentage composition** (fraction of each stc class within each cluster) rather than raw counts, so the three regions are directly comparable despite different dataset sizes.